# CS2 EXP-4 — CodeBERTa-small-v1 + LoRA


## 1. Dependencies


In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "peft<0.14.0",
    "accelerate",
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
], check=True)

print("Dependencies installed successfully for CUDA 12.1 driver!")

Dependencies installed successfully for CUDA 12.1 driver!


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"

DEVICE = "cuda:0"

print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")

Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 1.5 Settings


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"

WORKSPACE_ROOT = Path.cwd()

REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"

PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

SPLIT_ID = "cs1_project_holdout20_innercv_v1"

NORMALIZED_PARQUET = (
    PROCESSED_DIR
    / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"
)

OUTER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "outer_holdout"
    / "cs1_outer_project_holdout_manifest.parquet"
)

INNER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "inner_cv"
    / "cs1_project_grouped_5fold_manifest.parquet"
)

EXP4_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "case_study_2"
    / "exp4_codeberta_lora_v1"
)

EXP4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"
os.environ["HF_TOKEN"] = "secret"
RANK_GRID = (8, 16, 32)
EPOCHS = 4
SEARCH_EPOCHS = 2
SEARCH_N_SPLITS = 5

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 8

STORAGE_CAP_GB = 60

RUN_SMOKE_TEST = True
RUN_NESTED_OFFICIAL = True
RUN_CANONICAL_RETRAIN = True
RUN_HOLDOUT_EVAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
print(f"Device: {DEVICE}")


Settings loaded.
Workspace: /workspace
Repository: /workspace/DiverseVul--IS-Project
Data root: /workspace/IntelligentSystemProject/VulnerabilityDetectionData
Output root: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs
Hugging Face cache: /workspace/IntelligentSystemProject/hf_cache
Device: cuda:0


## 2. Clone the repository


In [4]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")
    
    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

   
    urllib.request.urlretrieve(zip_url, zip_path)

  
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

  
    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

 
    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")

Repository already exists at /workspace/DiverseVul--IS-Project


## 3. Verify GPU, RAM, and storage budget


In [5]:
import shutil
import psutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("EXP-4 requires a GPU runtime for LoRA fine-tuning.")

assert DEVICE == "cuda:0"

print("CUDA device:", torch.cuda.get_device_name(0))
print("bfloat16 supported:", torch.cuda.is_bf16_supported())
print("Total VRAM: %.2f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

print("CPU cores:", psutil.cpu_count(logical=True))
print("System RAM: %.1f GB" % (psutil.virtual_memory().total / 1e9))

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage already exceeds the {STORAGE_CAP_GB} GB storage cap for this machine.")


CUDA device: NVIDIA A100-SXM4-80GB
bfloat16 supported: True
Total VRAM: 84.99 GB
CPU cores: 112
System RAM: 2164.0 GB
Disk usage at /workspace: 5131.2 GB used / 5714.2 GB total (294.9 GB free)


## 4. Data availability check


In [6]:
required_data_files = {
    "normalized parquet": NORMALIZED_PARQUET,
    "outer holdout manifest": OUTER_MANIFEST_PATH,
    "inner CV manifest": INNER_MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by the Case Study 1 pipeline (normalization_v3.py + "
        "split_manifest.py) and were previously synced through Google Drive. On this machine, "
        "either:\n"
        "  1) Copy them from your previous Drive/Colab run into the paths above -- e.g. via the "
        "Jupyter file-browser upload, `scp`, or `rclone`/`gdown` from a terminal (you have root "
        "access here); or\n"
        "  2) Re-run the Case Study 1 notebook/pipeline against `data/raw/rdiversevul.json` to "
        "regenerate them locally.\n"
        f"Keep an eye on the {STORAGE_CAP_GB} GB storage cap while doing either."
    )
    raise FileNotFoundError("Required processed data/manifests are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


Found normalized parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet (210.7 MB)
Found outer holdout manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet (1.4 MB)
Found inner CV manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet (1.2 MB)


## 5. Write case_study_2 source files


In [7]:
(SRC_DIR / "case_study_2/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/__init__.py").write_text('')
print("Wrote", "case_study_2/__init__.py")


Wrote case_study_2/__init__.py


In [8]:
(SRC_DIR / "case_study_2/models.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/models.py").write_text('from __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoModel, AutoTokenizer\n\n\nDEFAULT_CODE_MODEL = "huggingface/CodeBERTa-small-v1"\nDEFAULT_CODE_TOKENIZER = "huggingface/CodeBERTa-small-v1"\n\n\ndef configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:\n    if hf_cache_dir:\n        hf_cache_dir = str(hf_cache_dir)\n        os.environ.setdefault("HF_HOME", hf_cache_dir)\n        os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path(hf_cache_dir) / "hub"))\n    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")\n    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")\n    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")\n    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")\n    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\n\ndef _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:\n    dtype_policy = (dtype_policy or "auto").lower()\n    device = str(device)\n    if dtype_policy == "float16":\n        return torch.float16 if device == "cuda" else torch.float32\n    if dtype_policy == "bfloat16":\n        return torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32\n    if dtype_policy == "float32":\n        return torch.float32\n    if dtype_policy == "auto":\n        if device == "cuda" and torch.cuda.is_bf16_supported():\n            return torch.bfloat16\n        if device == "cuda":\n            return torch.float32\n        return torch.float32\n    raise ValueError(f"Unknown dtype_policy: {dtype_policy}")\n\n\ndef load_code_tokenizer(\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,\n    hf_cache_dir: Optional[str] = None,\n):\n    configure_huggingface_cache(hf_cache_dir)\n    return AutoTokenizer.from_pretrained(\n        tokenizer_name,\n        use_fast=True,\n        cache_dir=hf_cache_dir,\n    )\n\n\ndef load_code_encoder(\n    model_name: str = DEFAULT_CODE_MODEL,\n    dtype_policy: str = "auto",\n    device: Optional[str] = None,\n    freeze: bool = True,\n    hf_cache_dir: Optional[str] = None,\n) -> nn.Module:\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    configure_huggingface_cache(hf_cache_dir)\n    dtype = _dtype_from_policy(dtype_policy, device)\n\n    kwargs: Dict[str, Any] = {"cache_dir": hf_cache_dir}\n    if dtype is not None:\n        kwargs["torch_dtype"] = dtype\n\n    model = AutoModel.from_pretrained(model_name, **kwargs)\n    model.to(device)\n\n    if freeze:\n        for param in model.parameters():\n            param.requires_grad = False\n        model.eval()\n\n    return model\n\n\ndef mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)\n    summed = (last_hidden_state * mask).sum(dim=1)\n    denom = mask.sum(dim=1).clamp(min=1.0)\n    return summed / denom\n\n\ndef cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:\n    return last_hidden_state[:, 0, :]\n\n\nclass CodeSequenceClassifier(nn.Module):\n    def __init__(\n        self,\n        model_name: str = DEFAULT_CODE_MODEL,\n        num_labels: int = 1,\n        freeze_backbone: bool = False,\n        pooling: str = "mean",\n        dtype_policy: str = "auto",\n        hf_cache_dir: Optional[str] = None,\n    ) -> None:\n        super().__init__()\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        self.backbone = load_code_encoder(\n            model_name=model_name,\n            dtype_policy=dtype_policy,\n            device=device,\n            freeze=freeze_backbone,\n            hf_cache_dir=hf_cache_dir,\n        )\n        hidden_size = int(self.backbone.config.hidden_size)\n        self.classification_head = nn.Linear(hidden_size, num_labels)\n        self.pooling = pooling\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:\n        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)\n        hidden = outputs.last_hidden_state\n        if self.pooling == "cls":\n            pooled = cls_pool_last_hidden(hidden)\n        else:\n            pooled = mean_pool_last_hidden(hidden, attention_mask)\n        logits = self.classification_head(pooled)\n        return logits.squeeze(-1)\n\n\ndef count_trainable_parameters(model: nn.Module) -> Dict[str, int]:\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    return {\n        "trainable_parameters": int(trainable),\n        "total_parameters": int(total),\n        "trainable_percent": float(100.0 * trainable / max(total, 1)),\n    }\n\n\ndef infer_lora_target_modules(model: nn.Module) -> List[str]:\n    module_names = [name for name, _ in model.named_modules()]\n    candidate_sets = [\n        ["qkv"],\n        ["q_proj", "v_proj"],\n        ["query", "value"],\n        ["in_proj"],\n    ]\n    for candidates in candidate_sets:\n        if all(any(name.endswith(candidate) or f".{candidate}" in name for name in module_names) for candidate in candidates):\n            return candidates\n    return ["query", "value"]\n\n\ndef create_lora_sequence_classifier(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    lora_dropout: float = 0.05,\n    pooling: str = "mean",\n    dtype_policy: str = "auto",\n    hf_cache_dir: Optional[str] = None,\n):\n    from peft import LoraConfig, get_peft_model\n\n    base = CodeSequenceClassifier(\n        model_name=model_name,\n        freeze_backbone=False,\n        pooling=pooling,\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n    )\n    target_modules = infer_lora_target_modules(base)\n    config = LoraConfig(\n        r=rank,\n        lora_alpha=lora_alpha,\n        target_modules=target_modules,\n        lora_dropout=lora_dropout,\n        bias="none",\n        task_type="FEATURE_EXTRACTION",\n    )\n    return get_peft_model(base, config)\n\n\ndef get_lora_model(model_name: str = DEFAULT_CODE_MODEL, rank: int = 8, lora_alpha: int = 16, pooling: str = "mean"):\n    return create_lora_sequence_classifier(\n        model_name=model_name,\n        rank=rank,\n        lora_alpha=lora_alpha,\n        pooling=pooling,\n    )\n')
print("Wrote", "case_study_2/models.py")


Wrote case_study_2/models.py


In [9]:
(SRC_DIR / "case_study_2/data_loader.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/data_loader.py").write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional, Dict, Any, List, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\n\n\nEMPTY_CODE_SENTINEL = "EMPTY_CODE_SAMPLE"\n\n\nclass CodeTextDataset(Dataset):\n    def __init__(\n        self,\n        dataframe: pd.DataFrame,\n        code_column: str = "normalized_code",\n        label_column: str = "label",\n        source_id_column: str = "source_row_id",\n        project_column: str = "project",\n    ) -> None:\n        self.df = dataframe.copy().reset_index(drop=True)\n        self.code_column = code_column\n        self.label_column = label_column\n        self.source_id_column = source_id_column\n        self.project_column = project_column\n\n        self.df[self.code_column] = self.df[self.code_column].fillna("").astype(str)\n        empty_mask = self.df[self.code_column].str.strip().eq("")\n        if empty_mask.any():\n            self.df.loc[empty_mask, self.code_column] = EMPTY_CODE_SENTINEL\n\n    def __len__(self) -> int:\n        return int(len(self.df))\n\n    def __getitem__(self, idx: int) -> Dict[str, Any]:\n        row = self.df.iloc[idx]\n        return {\n            "code": str(row[self.code_column]),\n            "label": int(row[self.label_column]),\n            "source_row_id": int(row[self.source_id_column]),\n            "project": str(row[self.project_column]),\n        }\n\n\n@dataclass\nclass TransformerBatchCollator:\n    tokenizer: Any\n    max_length: int = 512\n    pad_to_multiple_of: Optional[int] = 8\n\n    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:\n        texts = [feature["code"] for feature in features]\n        enc = self.tokenizer(\n            texts,\n            truncation=True,\n            max_length=self.max_length,\n            padding=True,\n            pad_to_multiple_of=self.pad_to_multiple_of,\n            return_tensors="pt",\n        )\n\n        labels = torch.tensor([feature["label"] for feature in features], dtype=torch.float32)\n        source_row_ids = torch.tensor([feature["source_row_id"] for feature in features], dtype=torch.long)\n        projects = [feature["project"] for feature in features]\n\n        enc["labels"] = labels\n        enc["label"] = labels\n        enc["source_row_id"] = source_row_ids\n        enc["project"] = projects\n        return enc\n\n\ndef create_dataloader(\n    dataframe: pd.DataFrame,\n    tokenizer: Any,\n    batch_size: int = 16,\n    max_length: int = 512,\n    shuffle: bool = False,\n    code_column: str = "normalized_code",\n    label_column: str = "label",\n    source_id_column: str = "source_row_id",\n    project_column: str = "project",\n    num_workers: int = 0,\n) -> DataLoader:\n    dataset = CodeTextDataset(\n        dataframe=dataframe,\n        code_column=code_column,\n        label_column=label_column,\n        source_id_column=source_id_column,\n        project_column=project_column,\n    )\n    collator = TransformerBatchCollator(\n        tokenizer=tokenizer,\n        max_length=max_length,\n        pad_to_multiple_of=8 if torch.cuda.is_available() else None,\n    )\n    return DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        drop_last=False,\n        num_workers=num_workers,\n        pin_memory=torch.cuda.is_available(),\n        collate_fn=collator,\n    )\n\n\ndef get_pos_weight(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    y = dataframe[label_column].astype(int).values\n    neg = int((y == 0).sum())\n    pos = int((y == 1).sum())\n    if pos == 0:\n        return torch.tensor([1.0], dtype=torch.float32)\n    return torch.tensor([neg / pos], dtype=torch.float32)\n\n\ndef get_class_weights(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:\n    return get_pos_weight(dataframe, label_column=label_column)\n\n\ndef sample_with_optional_positive_fraction(\n    frame: pd.DataFrame,\n    n_rows: int,\n    label_column: str = "label",\n    positive_fraction: Optional[float] = None,\n    random_state: int = 42,\n) -> pd.DataFrame:\n    if n_rows is None or n_rows <= 0 or len(frame) <= n_rows:\n        return frame.copy().reset_index(drop=True)\n\n    rng = np.random.default_rng(random_state)\n\n    if positive_fraction is None:\n        indices = rng.choice(frame.index.to_numpy(), size=n_rows, replace=False)\n        return frame.loc[indices].copy().reset_index(drop=True)\n\n    positives = frame[frame[label_column].astype(int) == 1]\n    negatives = frame[frame[label_column].astype(int) == 0]\n\n    n_pos = min(len(positives), max(1, int(round(n_rows * positive_fraction))))\n    n_neg = min(len(negatives), n_rows - n_pos)\n\n    pos_idx = rng.choice(positives.index.to_numpy(), size=n_pos, replace=False) if n_pos else []\n    neg_idx = rng.choice(negatives.index.to_numpy(), size=n_neg, replace=False) if n_neg else []\n    idx = np.concatenate([pos_idx, neg_idx])\n    rng.shuffle(idx)\n\n    return frame.loc[idx].copy().reset_index(drop=True)\n\n\ndef make_project_disjoint_threshold_split(\n    train_frame: pd.DataFrame,\n    threshold_fraction: float = 0.20,\n    project_column: str = "project",\n    label_column: str = "label",\n    random_state: int = 42,\n) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    projects = train_frame[[project_column, label_column]].groupby(project_column)[label_column].agg(["count", "sum"])\n    project_names = projects.index.to_numpy()\n\n    rng = np.random.default_rng(random_state)\n    shuffled = project_names.copy()\n    rng.shuffle(shuffled)\n\n    target_rows = int(round(len(train_frame) * threshold_fraction))\n    selected = []\n    count = 0\n\n    for project in shuffled:\n        selected.append(project)\n        count += int(projects.loc[project, "count"])\n        if count >= target_rows:\n            break\n\n    selected = set(selected)\n    threshold_mask = train_frame[project_column].isin(selected)\n    threshold_frame = train_frame[threshold_mask].copy().reset_index(drop=True)\n    fit_frame = train_frame[~threshold_mask].copy().reset_index(drop=True)\n\n    if (\n        fit_frame[label_column].sum() == 0\n        or threshold_frame[label_column].sum() == 0\n        or len(fit_frame) == 0\n        or len(threshold_frame) == 0\n    ):\n        shuffled_rows = train_frame.sample(frac=1.0, random_state=random_state).reset_index(drop=True)\n        cut = max(1, int(round(len(shuffled_rows) * (1.0 - threshold_fraction))))\n        fit_frame = shuffled_rows.iloc[:cut].copy().reset_index(drop=True)\n        threshold_frame = shuffled_rows.iloc[cut:].copy().reset_index(drop=True)\n\n    return fit_frame, threshold_frame\n')
print("Wrote", "case_study_2/data_loader.py")


Wrote case_study_2/data_loader.py


In [10]:
(SRC_DIR / "case_study_2/exp4/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp4/__init__.py").write_text('')
print("Wrote", "case_study_2/exp4/__init__.py")


Wrote case_study_2/exp4/__init__.py


In [11]:
(SRC_DIR / "case_study_2/exp4/exp4_lora.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp4/exp4_lora.py").write_text('from __future__ import annotations\n\nimport gc\nimport time\nimport torch\nimport torch.nn as nn\nfrom torch.optim import AdamW\nimport numpy as np\n\nfrom case_study_2.data_loader import create_dataloader, get_class_weights\nfrom case_study_2.models import get_lora_model, count_trainable_parameters, DEFAULT_CODE_MODEL\n\n\ndef train_lora_model(\n    train_df,\n    val_df,\n    tokenizer,\n    rank,\n    epochs=3,\n    batch_size=16,\n    grad_accum_steps=2,\n    eval_batch_size=32,\n    num_workers=2,\n    device="cuda",\n    hf_cache_dir=None,\n    code_column="normalized_code",\n    max_length=512,\n    verbose=True,\n    log_every_steps=50,\n    log_prefix="",\n):\n    train_loader = create_dataloader(\n        train_df, tokenizer, batch_size=batch_size, max_length=max_length,\n        shuffle=True, num_workers=num_workers, code_column=code_column,\n    )\n    val_loader = create_dataloader(\n        val_df, tokenizer, batch_size=eval_batch_size, max_length=max_length,\n        shuffle=False, num_workers=num_workers, code_column=code_column,\n    )\n\n    model = get_lora_model(model_name=DEFAULT_CODE_MODEL, rank=rank, lora_alpha=16).to(device)\n\n    is_cuda = (device == "cuda") or (hasattr(device, "type") and device.type == "cuda")\n\n    total_steps_per_epoch = -(-len(train_df) // batch_size)\n\n    if verbose:\n        stats = count_trainable_parameters(model)\n        print(\n            f"{log_prefix}[lora] rank={rank} | train_rows={len(train_df)} | val_rows={len(val_df)} | "\n            f"batch_size={batch_size} | grad_accum={grad_accum_steps} | steps/epoch={total_steps_per_epoch} | "\n            f"trainable={stats[\'trainable_parameters\']:,} ({stats[\'trainable_percent\']:.3f}%) | "\n            f"total={stats[\'total_parameters\']:,}"\n        )\n        if is_cuda:\n            print(f"{log_prefix}[lora] VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")\n\n    pos_weight = get_class_weights(train_df).to(device)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    optimizer = AdamW(model.parameters(), lr=2e-4)\n\n    t0 = time.time()\n\n    for epoch in range(epochs):\n        model.train()\n        epoch_loss = 0.0\n        n_steps = 0\n        epoch_t0 = time.time()\n        optimizer.zero_grad()\n\n        for step, batch in enumerate(train_loader):\n            input_ids = batch["input_ids"].to(device, non_blocking=True)\n            attention_mask = batch["attention_mask"].to(device, non_blocking=True)\n            labels = batch["label"].to(device, non_blocking=True)\n\n            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n                logits = model(input_ids, attention_mask)\n                loss = criterion(logits, labels) / grad_accum_steps\n\n            loss.backward()\n            epoch_loss += loss.item() * grad_accum_steps\n            n_steps += 1\n\n            if (step + 1) % grad_accum_steps == 0:\n                optimizer.step()\n                optimizer.zero_grad()\n\n            if verbose and log_every_steps and (step + 1) % log_every_steps == 0:\n                elapsed_min = (time.time() - epoch_t0) / 60\n                steps_left = total_steps_per_epoch - (step + 1)\n                rate = (step + 1) / max(elapsed_min, 1e-6)\n                eta_min = steps_left / max(rate, 1e-6)\n                print(\n                    f"{log_prefix}[lora] epoch {epoch+1}/{epochs} step {step+1}/{total_steps_per_epoch} | "\n                    f"avg_loss_so_far={epoch_loss/max(n_steps,1):.4f} | "\n                    f"elapsed={elapsed_min:.1f} min | ETA epoch ~{eta_min:.1f} min"\n                )\n\n        optimizer.step()\n        optimizer.zero_grad()\n\n        if verbose:\n            elapsed_min = (time.time() - t0) / 60\n            epoch_min = (time.time() - epoch_t0) / 60\n            peak_vram = torch.cuda.max_memory_allocated() / 1e9 if is_cuda else 0.0\n            print(\n                f"{log_prefix}[lora] epoch {epoch+1}/{epochs} done | avg_loss={epoch_loss/max(n_steps,1):.4f} "\n                f"| epoch_time={epoch_min:.1f} min | total_elapsed={elapsed_min:.1f} min | peak_VRAM={peak_vram:.2f} GB"\n            )\n\n    if verbose:\n        print(f"{log_prefix}[lora] training done, scoring validation set ({len(val_df)} rows)...")\n\n    model.eval()\n    all_scores = []\n    with torch.no_grad():\n        for batch in val_loader:\n            input_ids = batch["input_ids"].to(device, non_blocking=True)\n            attention_mask = batch["attention_mask"].to(device, non_blocking=True)\n            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):\n                logits = model(input_ids, attention_mask)\n                scores = torch.sigmoid(logits)\n            all_scores.extend(scores.float().cpu().numpy())\n\n    del train_loader, val_loader, criterion, optimizer\n    if is_cuda:\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n\n    return np.array(all_scores), model\n\n\ndef train_lora_model_safe(*args, max_retries=2, **kwargs):\n    batch_size = kwargs.pop("batch_size", 16)\n    grad_accum_steps = kwargs.pop("grad_accum_steps", 2)\n\n    attempt = 0\n    while True:\n        try:\n            return train_lora_model(\n                *args, batch_size=batch_size, grad_accum_steps=grad_accum_steps, **kwargs\n            )\n        except torch.cuda.OutOfMemoryError:\n            attempt += 1\n            gc.collect()\n            torch.cuda.empty_cache()\n            if attempt > max_retries or batch_size <= 2:\n                raise\n            new_batch_size = max(2, batch_size // 2)\n            new_grad_accum_steps = grad_accum_steps * max(1, batch_size // new_batch_size)\n            print(\n                f"[lora] CUDA OOM at batch_size={batch_size}; retrying "\n                f"(attempt {attempt}/{max_retries}) with batch_size={new_batch_size}, "\n                f"grad_accum_steps={new_grad_accum_steps} (effective batch size unchanged)."\n            )\n            batch_size, grad_accum_steps = new_batch_size, new_grad_accum_steps\n')
print("Wrote", "case_study_2/exp4/exp4_lora.py")


Wrote case_study_2/exp4/exp4_lora.py


In [12]:
(SRC_DIR / "case_study_2/exp4/exp4_nested_rank.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp4/exp4_nested_rank.py").write_text('from __future__ import annotations\n\nimport gc\nimport json\nimport time\nfrom dataclasses import dataclass, asdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, confusion_matrix\nimport matplotlib.pyplot as plt\n\nfrom case_study_2.data_loader import create_dataloader, get_class_weights\nfrom case_study_2.models import configure_huggingface_cache, load_code_tokenizer, DEFAULT_CODE_TOKENIZER\nfrom case_study_2.exp4.exp4_lora import train_lora_model_safe\nfrom case_study_1 import split_manifest\nfrom case_study_1 import evaluation\nfrom case_study_1.evaluation import EvaluationConfig\nfrom case_study_1.confidence_intervals import bootstrap_metric_ci, format_ci_report\n\n\nEXP4_VERSION = "cs2-exp4-codeberta-lora-v1"\n\n\n@dataclass(frozen=True)\nclass Exp4Config:\n    experiment_name: str = "cs2_exp4_codeberta_lora"\n\n    code_column: str = "normalized_code"\n    source_id_column: str = "source_row_id"\n    label_column: str = "label"\n    project_column: str = "project"\n    fold_column: str = "fold"\n\n    hf_cache_dir: Optional[str] = None\n    max_length: int = 512\n    train_batch_size: int = 16\n    grad_accum_steps: int = 2\n    epochs: int = 3\n\n    rank_grid: Tuple[int, ...] = (8, 16)\n    inner_n_splits: int = 3\n    inner_random_state: int = 20260707\n    decision_threshold: float = 0.50\n\n    search_epochs: int = 1\n    search_n_splits: int = 4\n\n    num_workers: int = 6\n    eval_batch_size: int = 64\n\n    n_splits: int = 5\n    random_state: int = 42\n    verbose: bool = True\n\n\ndef _checkpoint_paths(output_dir: Path, outer_fold_id: int) -> Dict[str, Path]:\n    root = output_dir / "checkpoints"\n    root.mkdir(parents=True, exist_ok=True)\n    prefix = f"outer_fold_{outer_fold_id}"\n    return {\n        "predictions": root / f"{prefix}_predictions.parquet",\n        "selected": root / f"{prefix}_selected_rank.json",\n        "training": root / f"{prefix}_outer_training.json",\n    }\n\n\ndef _search_checkpoint_path(output_dir: Path, outer_fold_id: int) -> Path:\n    root = output_dir / "checkpoints"\n    root.mkdir(parents=True, exist_ok=True)\n    return root / f"outer_fold_{outer_fold_id}_search_progress.json"\n\n\ndef _load_search_checkpoint(output_dir: Path, outer_fold_id: int) -> Dict[str, float]:\n    path = _search_checkpoint_path(output_dir, outer_fold_id)\n    if not path.exists():\n        return {}\n    with path.open("r", encoding="utf-8") as f:\n        raw = json.load(f)\n    return {int(k): float(v) for k, v in raw.items()}\n\n\ndef _save_search_checkpoint(output_dir: Path, outer_fold_id: int, rank_performance: Dict[int, float]) -> None:\n    path = _search_checkpoint_path(output_dir, outer_fold_id)\n    with path.open("w", encoding="utf-8") as f:\n        json.dump({str(k): v for k, v in rank_performance.items()}, f, indent=2)\n\n\ndef _write_outer_checkpoint(output_dir: Path, outer_fold_id: int, predictions: pd.DataFrame, selected: dict, training: dict) -> None:\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    predictions.to_parquet(paths["predictions"], index=False)\n    with paths["selected"].open("w", encoding="utf-8") as f:\n        json.dump(selected, f, indent=2, default=str)\n    with paths["training"].open("w", encoding="utf-8") as f:\n        json.dump(training, f, indent=2, default=str)\n\n\ndef _load_outer_checkpoint(output_dir: Path, outer_fold_id: int) -> Optional[dict]:\n    paths = _checkpoint_paths(output_dir, outer_fold_id)\n    if not all(p.exists() for p in paths.values()):\n        return None\n    with paths["selected"].open("r", encoding="utf-8") as f:\n        selected = json.load(f)\n    with paths["training"].open("r", encoding="utf-8") as f:\n        training = json.load(f)\n    return {\n        "predictions": pd.read_parquet(paths["predictions"]),\n        "selected": selected,\n        "training": training,\n    }\n\n\ndef _update_run_state(state_path: Path, completed_folds, status: str) -> None:\n    state = {\n        "status": status,\n        "updated_utc": datetime.now(timezone.utc).isoformat(),\n        "completed_outer_folds": sorted(int(f) for f in completed_folds),\n    }\n    with state_path.open("w", encoding="utf-8") as f:\n        json.dump(state, f, indent=2)\n\n\ndef run_exp4_nested_rank(\n    development_frame: pd.DataFrame,\n    development_manifest: pd.DataFrame,\n    config: Exp4Config,\n    output_dir: Path,\n    resume: bool = True,\n    additional_metadata: Optional[Dict[str, Any]] = None,\n) -> Dict[str, Any]:\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type != "cuda":\n        raise RuntimeError("EXP-4 LoRA fine-tuning requires a CUDA device.")\n\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    state_path = output_dir / "exp4_nested_run_state.json"\n\n    configure_huggingface_cache(config.hf_cache_dir)\n    tokenizer = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=config.hf_cache_dir)\n\n    fold_ids = sorted(development_manifest[config.fold_column].unique().tolist())\n    print(f"[nested] Starting EXP-4 nested rank search over {len(fold_ids)} outer folds, rank_grid={config.rank_grid}")\n    print(f"[nested] Search phase: {config.search_epochs} epoch(s), single held-out split | Refit phase: {config.epochs} epoch(s), full training")\n\n    oof_parts = []\n    selected_rows = []\n    outer_training_rows = []\n    completed_folds = []\n\n    t0 = time.time()\n\n    for outer_fold_id in fold_ids:\n        checkpoint = _load_outer_checkpoint(output_dir, outer_fold_id) if resume else None\n        if checkpoint is not None:\n            oof_parts.append(checkpoint["predictions"])\n            selected_rows.append(checkpoint["selected"])\n            outer_training_rows.append(checkpoint["training"])\n            completed_folds.append(outer_fold_id)\n            if config.verbose:\n                print(f"[nested] Outer fold {outer_fold_id}: loaded from checkpoint, skipping.")\n            continue\n\n        fold_t0 = time.time()\n        print(f"\\n=================== OUTER FOLD {outer_fold_id} ({len(completed_folds)+1}/{len(fold_ids)}) ===================")\n\n        outer_train_ids = development_manifest.loc[\n            development_manifest[config.fold_column] != outer_fold_id, config.source_id_column\n        ]\n        outer_val_ids = development_manifest.loc[\n            development_manifest[config.fold_column] == outer_fold_id, config.source_id_column\n        ]\n        outer_train_df = development_frame[development_frame[config.source_id_column].isin(outer_train_ids)].reset_index(drop=True)\n        outer_val_df = development_frame[development_frame[config.source_id_column].isin(outer_val_ids)].reset_index(drop=True)\n        print(f"[nested] outer_train={len(outer_train_df)} rows | outer_val={len(outer_val_df)} rows")\n\n        search_split_config = split_manifest.SplitConfig(\n            n_splits=config.search_n_splits,\n            random_state=config.inner_random_state,\n            shuffle=True,\n            source_id_column=config.source_id_column,\n            label_column=config.label_column,\n            group_column=config.project_column,\n        )\n        search_manifest = split_manifest.create_project_grouped_manifest(\n            outer_train_df[[config.source_id_column, config.label_column, config.project_column]],\n            config=search_split_config,\n        )\n        search_train_ids = search_manifest.loc[search_manifest["fold"] != 0, config.source_id_column]\n        search_val_ids = search_manifest.loc[search_manifest["fold"] == 0, config.source_id_column]\n        search_train_df = outer_train_df[outer_train_df[config.source_id_column].isin(search_train_ids)]\n        search_val_df = outer_train_df[outer_train_df[config.source_id_column].isin(search_val_ids)]\n        print(f"[nested] rank search split: train={len(search_train_df)} rows | val={len(search_val_df)} rows")\n\n        rank_performance = _load_search_checkpoint(output_dir, outer_fold_id) if resume else {}\n        if rank_performance:\n            print(f"[nested] resuming rank search, already have: {rank_performance}")\n\n        for rank_candidate in config.rank_grid:\n            if rank_candidate in rank_performance:\n                print(f"[nested] rank={rank_candidate}: already evaluated (PR-AUC={rank_performance[rank_candidate]:.4f}), skipping.")\n                continue\n\n            rank_t0 = time.time()\n            print(f"[nested] --- evaluating rank candidate {rank_candidate} ---")\n\n            val_scores, tmp_model = train_lora_model_safe(\n                search_train_df, search_val_df, tokenizer, rank=rank_candidate,\n                epochs=config.search_epochs, batch_size=config.train_batch_size,\n                grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,\n                num_workers=config.num_workers, device=device,\n                hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,\n                max_length=config.max_length, log_prefix="    ",\n            )\n            prauc = float(average_precision_score(search_val_df[config.label_column].values, val_scores))\n            rank_performance[rank_candidate] = prauc\n\n            print(f"[nested] rank={rank_candidate} | PR-AUC={prauc:.4f} | {(time.time()-rank_t0)/60:.1f} min")\n\n            _save_search_checkpoint(output_dir, outer_fold_id, rank_performance)\n\n            del tmp_model\n            gc.collect()\n            torch.cuda.empty_cache()\n\n        optimal_rank = max(rank_performance, key=rank_performance.get)\n        print(f"[nested] Selected rank={optimal_rank} for outer fold {outer_fold_id} | scores={rank_performance}")\n\n        print(f"[nested] --- final refit on full outer_train, rank={optimal_rank}, {config.epochs} epochs ---")\n        refit_t0 = time.time()\n        outer_val_scores, final_outer_model = train_lora_model_safe(\n            outer_train_df, outer_val_df, tokenizer, rank=optimal_rank,\n            epochs=config.epochs, batch_size=config.train_batch_size,\n            grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,\n            num_workers=config.num_workers, device=device,\n            hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,\n            max_length=config.max_length, log_prefix="    ",\n        )\n        print(f"[nested] refit done in {(time.time()-refit_t0)/60:.1f} min")\n\n        fold_oof = pd.DataFrame({\n            config.source_id_column: outer_val_df[config.source_id_column].values,\n            config.project_column: outer_val_df[config.project_column].values,\n            "label": outer_val_df[config.label_column].astype(int).values,\n            "y_score": outer_val_scores,\n            "fold": outer_fold_id,\n        })\n\n        selected_row = {"outer_fold_id": outer_fold_id, "selected_rank": optimal_rank, "search_scores": rank_performance}\n        training_row = {\n            "outer_fold_id": outer_fold_id,\n            "selected_rank": optimal_rank,\n            "n_train": int(len(outer_train_df)),\n            "n_val": int(len(outer_val_df)),\n            "elapsed_minutes": (time.time() - fold_t0) / 60,\n        }\n\n        if outer_fold_id == fold_ids[-1]:\n            final_outer_model.save_pretrained(output_dir / "final_exp4_lora_adapter")\n            print(f"[nested] saved final fold LoRA adapter to {output_dir / \'final_exp4_lora_adapter\'}")\n\n        _write_outer_checkpoint(output_dir, outer_fold_id, fold_oof, selected_row, training_row)\n\n        oof_parts.append(fold_oof)\n        selected_rows.append(selected_row)\n        outer_training_rows.append(training_row)\n        completed_folds.append(outer_fold_id)\n\n        _update_run_state(state_path, completed_folds, status="running")\n\n        del outer_train_df, outer_val_df, search_train_df, search_val_df, final_outer_model\n        gc.collect()\n        torch.cuda.empty_cache()\n\n        print(f"[nested] Outer fold {outer_fold_id} done in {training_row[\'elapsed_minutes\']:.1f} min | checkpoint saved | total elapsed {(time.time()-t0)/60:.1f} min")\n\n    oof_predictions = pd.concat(oof_parts, axis=0).reset_index(drop=True)\n\n    eval_config = EvaluationConfig(threshold=config.decision_threshold, expected_n_folds=len(fold_ids))\n    eval_results = evaluation.evaluate_oof_predictions(oof_predictions, config=eval_config)\n\n    selected_df = pd.DataFrame(selected_rows)\n    outer_training_df = pd.DataFrame(outer_training_rows)\n\n    artifacts = {\n        "oof_predictions": output_dir / "exp4_nested_oof_predictions.parquet",\n        "selected_rank_per_fold": output_dir / "exp4_selected_rank_per_outer_fold.csv",\n        "outer_training_audit": output_dir / "exp4_outer_training_audit.csv",\n        "run_metadata": output_dir / "exp4_nested_run_metadata.json",\n    }\n    oof_predictions.to_parquet(artifacts["oof_predictions"], index=False)\n    selected_df.to_csv(artifacts["selected_rank_per_fold"], index=False)\n    outer_training_df.to_csv(artifacts["outer_training_audit"], index=False)\n\n    metadata = {\n        "exp4_version": EXP4_VERSION,\n        "config": {**asdict(config), "rank_grid": list(config.rank_grid)},\n        "runtime_seconds": time.time() - t0,\n        **(additional_metadata or {}),\n    }\n    with open(artifacts["run_metadata"], "w", encoding="utf-8") as f:\n        json.dump(metadata, f, indent=2, default=str)\n\n    _update_run_state(state_path, completed_folds, status="completed")\n\n    print(f"\\n[nested] EXP-4 nested rank search complete in {(time.time()-t0)/60:.1f} min")\n\n    return {\n        "oof_predictions": oof_predictions,\n        "evaluation": eval_results,\n        "selected_rank": selected_df,\n        "outer_fold_training": outer_training_df,\n        "artifacts": artifacts,\n        "tokenizer": tokenizer,\n    }\n\n\ndef run_exp4_canonical_retrain(\n    development_frame: pd.DataFrame,\n    tokenizer,\n    selected_rank: int,\n    holdout_frame: pd.DataFrame,\n    config: Exp4Config,\n    output_dir: Path,\n) -> Dict[str, Any]:\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    print(f"[canonical] Retraining on full development set ({len(development_frame)} rows), rank={selected_rank}, {config.epochs} epochs")\n    t0 = time.time()\n\n    holdout_scores, global_model = train_lora_model_safe(\n        development_frame, holdout_frame, tokenizer, rank=selected_rank,\n        epochs=config.epochs, batch_size=config.train_batch_size,\n        grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,\n        num_workers=config.num_workers, device=device,\n        hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,\n        max_length=config.max_length, log_prefix="  ",\n    )\n    global_model.save_pretrained(output_dir / "final_canonical_lora_model")\n    print(f"[canonical] Done in {(time.time()-t0)/60:.1f} min | model saved to {output_dir / \'final_canonical_lora_model\'}")\n\n    holdout_predictions = pd.DataFrame({\n        config.source_id_column: holdout_frame[config.source_id_column].values,\n        config.project_column: holdout_frame[config.project_column].values,\n        "label": holdout_frame[config.label_column].astype(int).values,\n        "y_score": holdout_scores,\n        "fold": 0,\n    })\n\n    del global_model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    return {"holdout_predictions": holdout_predictions}\n\n\ndef run_exp4_holdout_evaluation(\n    holdout_predictions: pd.DataFrame,\n    config: Exp4Config,\n    output_dir: Path,\n    n_bootstrap: int = 1000,\n    confidence: float = 0.95,\n) -> Dict[str, Any]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    holdout_predictions.to_csv(output_dir / "exp4_holdout_predictions.csv", index=False)\n\n    eval_config = EvaluationConfig(threshold=config.decision_threshold, expected_n_folds=1)\n    holdout_metrics = evaluation.evaluate_oof_predictions(holdout_predictions, config=eval_config)\n    print(evaluation.format_metric_report(holdout_metrics["pooled_metrics"]))\n\n    ci_result = bootstrap_metric_ci(\n        holdout_predictions,\n        metric="average_precision_pr_auc",\n        group_column="project",\n        n_bootstrap=n_bootstrap,\n        confidence=confidence,\n        random_state=config.random_state,\n    )\n    with open(output_dir / "exp4_holdout_pr_auc_bootstrap_ci.json", "w", encoding="utf-8") as f:\n        json.dump(ci_result.as_dict(), f, indent=2)\n    print(format_ci_report(ci_result))\n\n    y_true = holdout_predictions["label"].values\n    y_score = holdout_predictions["y_score"].values\n    precision, recall, _ = precision_recall_curve(y_true, y_score)\n    ap = holdout_metrics["pooled_metrics"]["average_precision_pr_auc"]\n    plt.figure(figsize=(6, 5))\n    plt.plot(recall, precision, color="b", label=f"EXP-4 LoRA (PR-AUC = {ap:.4f})")\n    plt.xlabel("Recall")\n    plt.ylabel("Precision")\n    plt.title("Precision-Recall Curve - Frozen Outer Holdout")\n    plt.legend(loc="lower left")\n    plt.grid(True)\n    plt.savefig(output_dir / "exp4_outer_holdout_pr_curve.png")\n    plt.close()\n\n    y_pred = (y_score >= config.decision_threshold).astype(int)\n    cm = confusion_matrix(y_true, y_pred)\n    plt.figure(figsize=(4, 4))\n    plt.imshow(cm, cmap=plt.cm.Blues)\n    plt.title("Confusion Matrix - Frozen Outer Holdout")\n    plt.xlabel("Predicted")\n    plt.ylabel("Actual")\n    plt.xticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])\n    plt.yticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])\n    for i in range(2):\n        for j in range(2):\n            plt.text(j, i, str(cm[i, j]), ha="center", va="center")\n    plt.tight_layout()\n    plt.savefig(output_dir / "exp4_outer_holdout_confusion_matrix.png")\n    plt.close()\n\n    return {\n        "holdout_metrics": holdout_metrics,\n        "bootstrap_ci": ci_result.as_dict(),\n        "y_pred": y_pred,\n    }\n')
print("Wrote", "case_study_2/exp4/exp4_nested_rank.py")


Wrote case_study_2/exp4/exp4_nested_rank.py


## 6. Import project modules


In [13]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader, get_class_weights
from case_study_2.models import (
    configure_huggingface_cache,
    load_code_tokenizer,
    DEFAULT_CODE_MODEL,
    DEFAULT_CODE_TOKENIZER,
    count_trainable_parameters,
    CodeSequenceClassifier,
    infer_lora_target_modules,
)
from case_study_2.exp4.exp4_lora import train_lora_model, train_lora_model_safe
from case_study_2.exp4.exp4_nested_rank import (
    Exp4Config,
    run_exp4_nested_rank,
    run_exp4_canonical_retrain,
    run_exp4_holdout_evaluation,
)
from case_study_1.confidence_intervals import paired_bootstrap_metric_ci, format_paired_ci_report


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


## 7. Load dataset and frozen manifests


In [14]:
import pandas as pd

full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)

print("full_df rows:", len(full_df))
print("outer_manifest_df rows:", len(outer_manifest_df))
print("inner_manifest_df rows:", len(inner_manifest_df))


full_df rows: 261667
outer_manifest_df rows: 261667
inner_manifest_df rows: 203958


## 8. Build development and holdout frames


In [15]:
required_columns = {"source_row_id", "normalized_code", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if dev_ids != inner_ids:
    raise RuntimeError("Development partition and inner manifest coverage do not match.")
if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and holdout partitions overlap.")

development_frame = full_indexed.loc[full_indexed["source_row_id"].isin(dev_ids)].reset_index(drop=True)
holdout_frame = full_indexed.loc[full_indexed["source_row_id"].isin(holdout_ids)].reset_index(drop=True)

print("development_frame rows:", len(development_frame))
print("holdout_frame rows:", len(holdout_frame))


development_frame rows: 203958
holdout_frame rows: 57709


## 9. Configuration object


In [16]:
exp4_config = Exp4Config(
    hf_cache_dir=HF_CACHE_DIR,
    rank_grid=RANK_GRID,
    epochs=EPOCHS,
    search_epochs=SEARCH_EPOCHS,
    search_n_splits=SEARCH_N_SPLITS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    eval_batch_size=EVAL_BATCH_SIZE,
)
print(exp4_config)


Exp4Config(experiment_name='cs2_exp4_codeberta_lora', code_column='normalized_code', source_id_column='source_row_id', label_column='label', project_column='project', fold_column='fold', hf_cache_dir=PosixPath('/workspace/IntelligentSystemProject/hf_cache'), max_length=512, train_batch_size=32, grad_accum_steps=2, epochs=4, rank_grid=(8, 16, 32), inner_n_splits=3, inner_random_state=20260707, decision_threshold=0.5, search_epochs=2, search_n_splits=5, num_workers=8, eval_batch_size=64, n_splits=5, random_state=42, verbose=True)


## 10. Sanity-check LoRA target modules


In [17]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

_probe_model = CodeSequenceClassifier(model_name=DEFAULT_CODE_MODEL, freeze_backbone=False, dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred LoRA target_modules:", _target_modules)

del _probe_model
torch.cuda.empty_cache()


2026-08-02 16:22:20.404839: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-02 16:22:20.417423: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785687740.432193     911 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785687740.436999     911 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-02 16:22:20.455418: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Inferred LoRA target_modules: ['query', 'value']


## 11. Smoke test on a small subsample


In [18]:
import time
from sklearn.metrics import average_precision_score

if RUN_SMOKE_TEST:
    sample_df = development_frame.sample(n=min(2000, len(development_frame)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)

    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")

    t0 = time.time()
    smoke_scores, smoke_model = train_lora_model_safe(
        smoke_train, smoke_val, _tok_check, rank=RANK_GRID[0], epochs=1,
        batch_size=exp4_config.train_batch_size, grad_accum_steps=exp4_config.grad_accum_steps,
        eval_batch_size=exp4_config.eval_batch_size, num_workers=exp4_config.num_workers,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=exp4_config.code_column,
        max_length=exp4_config.max_length,
    )
    smoke_prauc = float(average_precision_score(smoke_val[exp4_config.label_column].values, smoke_scores))
    print(f"[smoke] PR-AUC={smoke_prauc:.4f} | elapsed={(time.time()-t0)/60:.1f} min")

    del smoke_model
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


[smoke] train=1500 rows | val=500 rows
[lora] rank=8 | train_rows=1500 | val_rows=500 | batch_size=32 | grad_accum=2 | steps/epoch=47 | trainable=147,456 (0.176%) | total=83,599,105
[lora] epoch 1/1 done | avg_loss=1.2683 | epoch_time=0.1 min | total_elapsed=0.1 min | peak_VRAM=0.00 GB
[lora] training done, scoring validation set (500 rows)...
[smoke] PR-AUC=0.0853 | elapsed=0.3 min


## 12. Official nested LoRA rank search


In [19]:
if RUN_NESTED_OFFICIAL:
    nested_results = run_exp4_nested_rank(
        development_frame=development_frame,
        development_manifest=inner_manifest_df,
        config=exp4_config,
        output_dir=EXP4_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(NORMALIZED_PARQUET),
            "outer_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
        },
    )
    display(nested_results["selected_rank"])
    print("Pooled nested PR-AUC:", nested_results["evaluation"]["pooled_metrics"]["average_precision_pr_auc"])
    tokenizer = nested_results["tokenizer"]
else:
    nested_results = None
    tokenizer = _tok_check
    print("RUN_NESTED_OFFICIAL=False; skipping.")


[nested] Starting EXP-4 nested rank search over 5 outer folds, rank_grid=(8, 16, 32)
[nested] Search phase: 2 epoch(s), single held-out split | Refit phase: 4 epoch(s), full training

=================== OUTER FOLD 0 (1/5) ===================
[nested] outer_train=148449 rows | outer_val=55509 rows
[nested] rank search split: train=109885 rows | val=38564 rows
[nested] --- evaluating rank candidate 8 ---
    [lora] rank=8 | train_rows=109885 | val_rows=38564 | batch_size=32 | grad_accum=2 | steps/epoch=3434 | trainable=147,456 (0.176%) | total=83,599,105
    [lora] VRAM after model load: 0.19 GB
    [lora] epoch 1/2 step 50/3434 | avg_loss_so_far=1.2504 | elapsed=0.0 min | ETA epoch ~3.3 min
    [lora] epoch 1/2 step 100/3434 | avg_loss_so_far=1.2018 | elapsed=0.1 min | ETA epoch ~3.0 min
    [lora] epoch 1/2 step 150/3434 | avg_loss_so_far=1.1616 | elapsed=0.1 min | ETA epoch ~2.8 min
    [lora] epoch 1/2 step 200/3434 | avg_loss_so_far=1.1630 | elapsed=0.2 min | ETA epoch ~2.8 min
   

,outer_fold_id,selected_rank,search_scores
0,0,16,"{8: 0.13763073941370582, 16: 0.151849881908121..."
1,1,32,"{8: 0.04712301587301587, 16: 0.055052790346907..."
2,2,32,"{8: 0.09513665658101945, 16: 0.079562908522913..."
3,3,16,"{8: 0.3089369415414307, 16: 0.4602503501400560..."
4,4,32,"{8: 0.28068181818181814, 16: 0.246428571428571..."


Pooled nested PR-AUC: 0.11635366380516732


## 13. Canonical retrain and frozen outer holdout scoring


In [20]:
if RUN_CANONICAL_RETRAIN and nested_results is not None:
    global_selected_rank = int(nested_results["selected_rank"]["selected_rank"].mode()[0])
    retrain_results = run_exp4_canonical_retrain(
        development_frame=development_frame,
        tokenizer=tokenizer,
        selected_rank=global_selected_rank,
        holdout_frame=holdout_frame,
        config=exp4_config,
        output_dir=EXP4_OUTPUT_DIR,
    )
    print("Canonical model trained with rank =", global_selected_rank)
else:
    retrain_results = None
    print("RUN_CANONICAL_RETRAIN=False or no nested results; skipping.")


[canonical] Retraining on full development set (203958 rows), rank=32, 4 epochs
  [lora] rank=32 | train_rows=203958 | val_rows=57709 | batch_size=32 | grad_accum=2 | steps/epoch=6374 | trainable=589,824 (0.702%) | total=84,041,473
  [lora] VRAM after model load: 0.19 GB
  [lora] epoch 1/4 step 50/6374 | avg_loss_so_far=1.2858 | elapsed=0.1 min | ETA epoch ~6.4 min
  [lora] epoch 1/4 step 100/6374 | avg_loss_so_far=1.3015 | elapsed=0.1 min | ETA epoch ~5.7 min
  [lora] epoch 1/4 step 150/6374 | avg_loss_so_far=1.2682 | elapsed=0.1 min | ETA epoch ~5.5 min
  [lora] epoch 1/4 step 200/6374 | avg_loss_so_far=1.2304 | elapsed=0.2 min | ETA epoch ~5.3 min
  [lora] epoch 1/4 step 250/6374 | avg_loss_so_far=1.2089 | elapsed=0.2 min | ETA epoch ~5.2 min
  [lora] epoch 1/4 step 300/6374 | avg_loss_so_far=1.2115 | elapsed=0.3 min | ETA epoch ~5.1 min
  [lora] epoch 1/4 step 350/6374 | avg_loss_so_far=1.2050 | elapsed=0.3 min | ETA epoch ~5.1 min
  [lora] epoch 1/4 step 400/6374 | avg_loss_so_far

## 14. Frozen outer holdout evaluation with bootstrap confidence interval


In [21]:
if RUN_HOLDOUT_EVAL and retrain_results is not None:
    holdout_results = run_exp4_holdout_evaluation(
        holdout_predictions=retrain_results["holdout_predictions"],
        config=exp4_config,
        output_dir=EXP4_OUTPUT_DIR,
    )
else:
    holdout_results = None
    print("RUN_HOLDOUT_EVAL=False or no canonical retrain results; skipping.")


Pooled Out-of-Fold Evaluation
                   n_samples: 57709
                vulnerable_1: 3211
            non_vulnerable_0: 54498
               positive_rate: 0.055641
                   threshold: 0.500000
    average_precision_pr_auc: 0.134952
                   precision: 0.095697
                      recall: 0.691996
                          f1: 0.168142
                         mcc: 0.143377
                 specificity: 0.614720
         false_positive_rate: 0.385280
               true_negative: 33501
              false_positive: 20997
              false_negative: 989
               true_positive: 2222
average_precision_pr_auc: point estimate = 0.1350
  95% CI (project-block bootstrap): [0.1113, 0.1658]
  valid resamples: 1000/1000 (0 degenerate, dropped)
  n_projects: 203, random_state=42
  Reflects sampling variability within this dataset only; not an estimate of generalization to C functions outside this collection.


## 15. Paired comparison against EXP-3 on the frozen holdout


In [22]:
EXP3_HOLDOUT_PREDICTIONS_PATH = OUTPUT_ROOT / "case_study_2" / "exp3_codeberta_linear_probe_v1" / "exp3_holdout_predictions.csv"

if RUN_HOLDOUT_EVAL and retrain_results is not None and EXP3_HOLDOUT_PREDICTIONS_PATH.exists():
    exp3_holdout = pd.read_csv(EXP3_HOLDOUT_PREDICTIONS_PATH)
    comparison = paired_bootstrap_metric_ci(
        predictions_a=retrain_results["holdout_predictions"],
        predictions_b=exp3_holdout,
        experiment_name_a="EXP-4 LoRA",
        experiment_name_b="EXP-3 linear probe",
        metric="average_precision_pr_auc",
    )
    print(format_paired_ci_report(comparison))
else:
    print("Run EXP-4 holdout evaluation and the EXP-3 notebook first.")


Run EXP-4 holdout evaluation and the EXP-3 notebook first.


## 16. Cleanup


In [23]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP4_OUTPUT_DIR}.")


VRAM allocated: 0.01703936 GB
Disk usage at /workspace: 5137.6 GB used / 5714.2 GB total (288.6 GB free)
